# ⚙️ SHAPE Agent 16 — Section 4: System Integration & Deployment
## Authored by: **Michael Rivas**
### ITAI2376 · Section 14562 · Houston Community College

---

| | |
|---|---|
| **Author** | Michael Rivas |
| **Role** | System Architecture, Integration & Project Coordination |
| **Design Contribution** | SQLite schema, 8-step ReAct orchestrator, multi-agent persona system, mock mode, full deployment, 4-Step Memory Demo Protocol integration |
| **Coding** | `shape_agent_core.py`, `shape_db_handler.py`, `shape_gradio_ui.py`, `shape_rpg_app.py`, `shape_memory.py`, `shape_consolidate.py`, `shape_recall_probe.py`, `shape_stability_check.py`, `setup.sh`, `run.sh` |
| **Reflection** | See Section 8 of this notebook |

**Full team:** Michael Rivas · Cherluk · Olu · Taki

---

## What this notebook covers

This is **Section 4 of 4** — the integration and deployment section.  
I assembled all components from Sections 1–3 into the final SHAPE Agent 16 system,  
designed the database schema, built the 8-step orchestrator, deployed both UIs,  
and ran the complete **4-Step Episodic Memory Demo Protocol** that ties all teammates' work  
into a cross-session memory lifecycle.

**Depends on:** All three previous sections (Cherluk, Olu, Taki).

---

**Filename:** `NB_MichaelRivas_Section4_Integration_Group_ITAI2376_14562.ipynb`

In [ ]:
# ── Setup & Team API Key Loader ───────────────────────────────────────────────
# Michael: I set up the shared key system so teammates don't need their own API key.
# The key loads from config/groq_key.txt (committed to repo), Colab Secrets,
# or falls back to mock mode automatically.

!pip install -q groq gradio streamlit python-dotenv pyngrok

import os, sys
from pathlib import Path

REPO_URL = 'https://github.com/mrivassnj-svg/SHAPE_Agent16.git'
REPO_DIR = '/content/SHAPE_Agent16'
if not Path(REPO_DIR).exists():
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull --quiet
os.chdir(REPO_DIR)
sys.path.insert(0, 'src')

def _load_key():
    if os.environ.get('GROQ_API_KEY','').startswith('gsk_'): return os.environ['GROQ_API_KEY']
    kf = Path('config/groq_key.txt')
    if kf.exists():
        k = kf.read_text().strip()
        if k.startswith('gsk_'): return k
    try:
        from google.colab import userdata
        k = userdata.get('GROQ_API_KEY') or ''
        if k.startswith('gsk_'): return k
    except: pass
    return ''

key = _load_key()
if key:
    os.environ['GROQ_API_KEY']     = key
    os.environ['SHAPE_MOCK_MODE']  = ''
    print('✅ API key loaded — live LLM mode.')
else:
    os.environ['SHAPE_MOCK_MODE'] = '1'
    print('🔧 Mock mode — LLM responses simulated. All architecture demos still work.')

---
## 1. System Architecture

**Author: Michael Rivas**

```
┌─────────────────────────────────────────────────────────────┐
│              Gradio UI          Streamlit RPG                │
│          (Chat + Lessons)    (Python Battle Game)            │
└──────────────┬──────────────────────────┬───────────────────┘
               │ user turn                │ battle turn
               ▼                          ▼
┌──────────────────────────────────────────────────────────────┐
│                shape_agent_core.py                           │
│                8-Step ReAct Orchestrator                     │
│                                                              │
│  Step 1 RECEIVE  → Safety gate (Olu)                         │
│  Step 2 PROFILE  → Load learner + distilled memory (Michael) │
│  Step 3 RETRIEVE → Tool 2 curriculum search (Olu+Cherluk)    │
│  Step 4 REASON   → CoT + ε-greedy strategy (Taki)           │
│  Step 5 ACT      → Tool 1 code runner if needed (Olu)        │
│  Step 6 RESPOND  → LLM call with context (Michael)           │
│  Step 7 EVALUATE → Reward + Q-update (Taki)                  │
│  Step 8 ADAPT    → Level promotion + topic tracking (Michael)│
│                                                              │
│  Sub-agents: Instructor · Debugger · Evaluator · Strategist  │
└───────┬──────────────────────┬───────────────────────────────┘
        │                      │
        ▼                      ▼
┌──────────────┐    ┌──────────────────────┐
│ shape_tools  │    │  shape_rl_policy     │
│  Tool 1 (Olu)│    │  ε-greedy Q-table    │
│  Tool 2 (Olu)│    │  (Taki)              │
└──────┬───────┘    └──────────┬───────────┘
       │                       │
       └───────────►  shape_db_handler.py  ◄─── shape_rpg_db.py
                      SQLite: shape.db
                      (Michael)
                           │
                           ▼
              shape_memory.py   shape_consolidate.py
              episodes table    distilled_memory table
              (Michael)         (Michael)
```

### Database Schema — 11 tables

| Table | Owner | Purpose |
|---|---|---|
| `learners` | Michael | Profiles — level, topics seen |
| `sessions` | Michael | One row per conversation |
| `interactions` | Michael | Every turn, strategy used, reward |
| `assessments` | Taki | Pre/post test scores |
| `curriculum` | Cherluk | Instructor-authored lessons |
| `policy_stats` | Taki | Per-strategy Q-values |
| `rpg_players` | Michael | HP, XP, level, zone, gold |
| `rpg_battles` | Michael | Enemy encounters, outcomes |
| `rpg_battle_turns` | Michael | Per-question battle log |
| `episodes` | Michael | Raw episodic memory log (wake phase) |
| `distilled_memory` | Michael | Consolidated cross-session memory |

In [ ]:
# ── 2. Database Initialisation & Verification ─────────────────────────────────
# Michael: I designed the full SQLite schema that ties all components together.

import sqlite3
import shape_db_handler as db
from shape_rpg_db import init_rpg_tables
from shape_main import seed_database

db.init_db()
init_rpg_tables()
seed_database()

DB_PATH = Path('data/shape.db')
conn = sqlite3.connect(str(DB_PATH))
tables = conn.execute("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name").fetchall()
conn.close()

print('DATABASE SCHEMA (Michael Rivas)')
print('='*55)
print(f'\nshape.db — {len(tables)} tables:\n')
OWNERS = {
    'learners':'Michael','sessions':'Michael','interactions':'Michael',
    'assessments':'Taki','curriculum':'Cherluk','policy_stats':'Taki',
    'rpg_players':'Michael','rpg_battles':'Michael','rpg_battle_turns':'Michael',
    'episodes':'Michael','distilled_memory':'Michael',
}
for (name,) in tables:
    owner = OWNERS.get(name,'?')
    print(f'  📋 {name:<22} (designed by {owner})')
print(f'\n✅ All {len(tables)} tables ready.')

In [ ]:
# ── 3. 8-Step ReAct Orchestrator — Live Demo ──────────────────────────────────
# Michael: I assembled Cherluk's curriculum + Olu's tools + Taki's RL policy
# into a single coherent reasoning cycle.

import uuid
from shape_agent_core import run_turn

print('8-STEP REACT ORCHESTRATOR — Live Walkthrough (Michael Rivas)')
print('='*55)

db.upsert_learner('michael_demo', 'Michael Rivas')
session = str(uuid.uuid4())

inputs = [
    ('Turn 1 — normal question',    'What is a for loop in Python?'),
    ('Turn 2 — answer submission',  'Check my answer: a for loop uses the range() function'),
    ('Turn 3 — code submission',    'Does this work?\n```python\nfor i in range(5):\n    print(i*2)\n```'),
    ('Turn 4 — broken code',        '```python\nfor i in range(5)\n    print(i)\n```'),
]

for turn_num, (label, user_input) in enumerate(inputs, 1):
    print(f'\n── {label} ──')
    print(f'Input: {user_input[:80]}...' if len(user_input)>80 else f'Input: {user_input}')
    reply = run_turn('michael_demo', user_input, session, turn_num)
    print(f'Reply: {reply[:200]}...' if len(reply) > 200 else f'Reply: {reply}')

In [ ]:
# ── 4. Multi-Agent System — All 4 Personas ────────────────────────────────────
# Michael: I implemented 4 sub-agent personas via dynamic system prompt injection.

print('MULTI-AGENT SYSTEM (Michael Rivas)')
print('='*55)
print()
print('Persona selection logic (in shape_agent_core.py):')
print('  code fence detected?    → DEBUGGER')
print('  "check my answer"?      → EVALUATOR')
print('  default                 → INSTRUCTOR')
print('  every 5 turns internally→ STRATEGIST (level recommendation)')
print()

session2 = str(uuid.uuid4())
persona_demos = [
    ('INSTRUCTOR  (default question)',
     'What is a Python dictionary?'),
    ('EVALUATOR   (answer check keyword)',
     'Check my answer: a dictionary maps keys to values using curly braces.'),
    ('DEBUGGER    (broken code)',
     'Fix this:\n```python\nmy_dict = {"a":1, "b":2\nprint(my_dict)\n```'),
]

for persona, message in persona_demos:
    print(f'\n🤖 {persona}')
    reply = run_turn('michael_demo', message, session2, persona_demos.index((persona, message))+1)
    print(f'   {reply[:250].replace(chr(10), " | ")}...')

In [ ]:
# ── 5. Learner Adaptation & Memory ────────────────────────────────────────────
# Michael: Shows the persistent learner profile and level promotion system.

import json
from shape_db_handler import get_learner, get_assessments
from shape_rl_policy import policy_summary

print('LEARNER ADAPTATION SYSTEM (Michael Rivas)')
print('='*55)

learner = get_learner('michael_demo')
topics  = json.loads(learner['topics_seen'])

print(f'\nLearner:     {learner["display_name"]}')
print(f'Level:       {learner["level"]}')
print(f'Topics seen: {topics if topics else "(interact more to populate)"}')
print()
print('Level promotion rule (_adapt_learner in shape_agent_core.py):')
print('  IF last 3 agent turns all have reward = 1.0 (correct, no hints):')
print('    beginner      → intermediate')
print('    intermediate  → advanced')
print()
print('Current RL policy state (Taki\'s Q-table):')
print(policy_summary())

---
## 6. 4-Step Episodic Memory Demo Protocol

**Author: Michael Rivas**

I assembled all teammates' memory contributions into a runnable end-to-end demo  
that proves the system can remember, consolidate, recall, and self-certify stability  
across completely separate sessions.

```
Session A  ──► Cherluk: log_episode()  (Step 1 — Wake Phase)
                │
                ▼
            Michael: run_consolidation()  (Step 2 — Offline Sleep Job)
                │
                ▼
Session B  ──► Taki:   run_all_probes()  (Step 3 — Cross-Session Recall)
                │
                ▼
            Michael: run_stability_check()  (Step 4 — Anti-Forgetting)
```

Each step runs against `shape.db` and all results are deterministic (no LLM required).  
The demo uses learner id `memory_demo_learner` as a fresh subject.

In [ ]:
# ── Step 1: Wake Phase (Cherluk's contribution) ───────────────────────────────
# Michael: I wire Cherluk's episode types into a multi-turn session demo.

import uuid
from shape_memory import log_episode, get_active_episodes

LEARNER = 'memory_demo_learner'
db.upsert_learner(LEARNER, 'Demo Learner')

print('STEP 1 — WAKE PHASE (Cherluk)')
print('='*55)

# Session A — initial session with facts, preferences, distractors
sid_A = str(uuid.uuid4())
db.create_session(sid_A, LEARNER)

turns = [
    (1, 'fact',       'project_name',       'SHAPE Agent 16',              'high'),
    (2, 'preference', 'preferred_language',  'Python',                      'high'),
    (3, 'skill',      'skill_for_loops',     'Completed for-loop exercise', 'high'),
    (4, 'distractor', 'weather',             'Mentioned it is cold outside','medium'),
    (5, 'fact',       'course_code',         'ITAI2376',                    'high'),
    # Turn 6: mid-session correction — preferred_language updated
    (6, 'correction', 'preferred_language',  'JavaScript (corrected)',      'high'),
]

eids = {}
for turn_num, ep_type, subject, content, confidence in turns:
    eid = log_episode(sid_A, LEARNER, turn_num,
                      episode_type=ep_type, subject=subject,
                      content=content, confidence=confidence)
    eids[subject] = eid
    icon = '🔄' if ep_type == 'correction' else '📝'
    print(f'  {icon} Turn {turn_num} [{ep_type:11s}] {subject}: "{content}"')

active = get_active_episodes(LEARNER)
print(f'\n  Active episodes after session A: {len(active)}')
for ep in active:
    print(f'    id={ep["episode_id"]}  salience={ep["salience"]:.2f}  [{ep["episode_type"]:11s}]  {ep["subject"]}')

print()
print('✅ Step 1 complete — preferred_language "Python" superseded by "JavaScript"')

In [ ]:
# ── Step 2: Offline Consolidation / Sleep Job (Michael) ──────────────────────
# Michael: I run the consolidation job that distills high-salience episodes
# into distilled_memory and prunes ephemeral session chatter.

from shape_consolidate import run_consolidation
from shape_memory import get_distilled_memory

print('STEP 2 — OFFLINE CONSOLIDATION (Michael)')
print('='*55)

report = run_consolidation(LEARNER)
print()
print(report.summary())

print()
print('Distilled memory after consolidation:')
mems = get_distilled_memory(LEARNER)
for m in mems:
    print(f'  📌 [{m["confidence"]:6s}] {m["subject"]:25s} → "{m["content"][:50]}"')

print()
lang_mem = next((m for m in mems if m['subject'] == 'preferred_language'), None)
if lang_mem:
    if 'JavaScript' in lang_mem['content']:
        print('✅ Correction applied — distilled memory holds "JavaScript", not "Python"')
    else:
        print('⚠️  Check: distilled memory may hold stale value')

In [ ]:
# ── Step 3: Cross-Session Recall (Taki's contribution — integrated by Michael) ──
# Michael: I call Taki's recall probe suite in a fresh context (zero preamble).
# This simulates session B — a completely new session with no conversation history.

from shape_recall_probe import run_all_probes, print_probe_report

print('STEP 3 — CROSS-SESSION RECALL (Taki probe suite, run by Michael)')
print('='*55)
print()
print('Simulating Session B — zero prompt preamble, fresh session context.')
print('The agent must recall via distilled_memory only — no conversation history.')
print()

results = run_all_probes(LEARNER)
print_probe_report(results)

passed = sum(1 for r in results if r.passed())
print(f'Recall probe result: {passed}/{len(results)} probes passed')

In [ ]:
# ── Step 4: Stability / Anti-Forgetting Check (Michael) ───────────────────────
# Michael: I run the full stability check to certify that consolidation
# improved memory without degrading any agent capability.

from shape_stability_check import run_stability_check

print('STEP 4 — STABILITY / ANTI-FORGETTING (Michael)')
print('='*55)

stability_report = run_stability_check(LEARNER)
print(stability_report.summary())

if stability_report.overall_stable:
    print('🟢 SYSTEM STABLE — memory consolidation passed all capability checks.')
else:
    print('🔴 SYSTEM UNSTABLE — review failed checks before the next session.')

In [ ]:
# ── 7. Launch Learning Companion — Gradio UI ──────────────────────────────────
# Michael: I built the Gradio interface with 3 tabs:
#   Tab 1 — 💬 Chat (all 4 agent personas)
#   Tab 2 — 📝 Lesson Builder (Cherluk/Olu/Taki add content without coding)
#   Tab 3 — 📈 RL Policy Monitor (Taki's Q-table live view)
#
# share=True creates a public gradio.live URL valid for 72 hours.
# Share the URL with your team for the demo video!

from shape_gradio_ui import build_app
print('Launching SHAPE Agent 16 Learning Companion...')
print('A public URL will appear below — share it for the demo video!')
print()
app = build_app()
app.launch(share=True, server_port=7860, quiet=False)

In [ ]:
# ── 8. Launch Python RPG — Streamlit ──────────────────────────────────────────
# Michael: I built the Streamlit RPG battle game.

import subprocess, time

print('Launching SHAPE Agent 16 RPG...')
rpg_proc = subprocess.Popen(
    ['streamlit', 'run', 'src/shape_rpg_app.py',
     '--server.port', '8501', '--server.headless', 'true',
     '--theme.base', 'dark', '--theme.primaryColor', '#e94560',
     '--theme.backgroundColor', '#1a1a2e',
     '--theme.secondaryBackgroundColor', '#16213e'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)
time.sleep(5)

try:
    from pyngrok import ngrok
    tunnel = ngrok.connect(8501)
    print(f'\n⚔️  RPG is LIVE at: {tunnel.public_url}')
    print('Share this URL with your team!')
except Exception as e:
    print(f'RPG at http://localhost:8501  (ngrok: {e})')

In [ ]:
# ── 9. Final Submission Checklist ─────────────────────────────────────────────
# Michael: Final verification before submitting on Canvas.

print('ITAI2376 SUBMISSION CHECKLIST (Michael Rivas)')
print('='*55)

items = [
    ('✅','Option 4: Interactive Learning Companion'),
    ('✅','One LLM agent minimum (Groq Qwen 27B)'),
    ('✅','4 sub-agents: Instructor, Debugger, Evaluator, Strategist'),
    ('✅','ReAct pattern — 8-step reasoning cycle'),
    ('✅','Chain-of-Thought scratchpad per turn'),
    ('✅','Tool 1: Python exercise runner (Olu)'),
    ('✅','Tool 2: Curriculum keyword retrieval (Olu + Cherluk)'),
    ('✅','RL: ε-greedy contextual bandit (Taki)'),
    ('✅','RL: Reward function defined (Taki)'),
    ('✅','RL: Q-values persist in SQLite (Michael + Taki)'),
    ('✅','Safety: Prompt injection detection (Olu)'),
    ('✅','Safety: AST code sandbox + timeout (Olu)'),
    ('✅','Safety: Fallback + scope enforcement (Olu)'),
    ('✅','Safety: Memory store protection — 2-layer defense (Olu + Michael)'),
    ('✅','Memory: SQLite learner profiles, history, topics (Michael)'),
    ('✅','Memory: Episodic encoding with salience/confidence (Michael + Cherluk)'),
    ('✅','Memory: Offline consolidation / sleep job (Michael)'),
    ('✅','Memory: Cross-session recall probes — direct, implicit, contradiction (Taki + Michael)'),
    ('✅','Memory: Stability / anti-forgetting check (Taki + Michael)'),
    ('✅','Evaluation metrics with formulas (Taki)'),
    ('✅','Gradio UI + Lesson Builder (Michael + Cherluk)'),
    ('✅','Streamlit RPG battle game (Michael)'),
    ('✅','Blender sprite generator (Michael)'),
    ('✅','All 4 names on every deliverable'),
    ('✅','4 individual notebooks (Cherluk, Olu, Taki, Michael)'),
    ('✅','FN_Proposal_MichaelRivas_Group_ITAI2376_14562.pdf'),
    ('✅','README.md + requirements.txt'),
    ('✅','Team Colab notebook (SHAPE_Agent16_Colab.ipynb)'),
    ('✅','75 pytest tests — 100% passing'),
    ('⬜','GitHub repo link → Canvas submission'),
    ('⬜','FN_Report_MichaelRivas_Group_ITAI2376_14562.pdf — due Dec 7'),
    ('⬜','FN_Demo_MichaelRivas_Group_ITAI2376_14562.mp4  — due Dec 7'),
]

done = sum(1 for i,_ in items if i == '✅')
todo = sum(1 for i,_ in items if i == '⬜')
for icon, label in items:
    print(f'  {icon}  {label}')
print(f'\n{done} complete · {todo} remaining')

---
## 8. Reflection — Michael Rivas

**Architecture decisions I made:**

**SQLite over a hosted database** — zero external dependencies. The DB file travels with the repo, opens in Colab, runs on any laptop. No connection strings, no credentials, no cloud setup.

**`ToolResult` dataclass (Olu's interface)** — I required Olu to return a typed object from every tool. This meant I could write the orchestrator's branching logic once and add new tools without touching the core loop.

**System prompt injection for multi-agent** — Four sub-agents don't require four LLM calls or AutoGen. I parameterised a single system prompt template with `{level}` and `{strategy}`, selected among four role prompts per turn, and made one Groq call. Same cost, full flexibility.

**Mock mode** — After Cherluk, Olu, and Taki finished their sections, I noticed none of them had API keys. I added `SHAPE_MOCK_MODE=1` with realistic canned responses so all three could run every demo cell without blocking on the LLM.

**Episodic memory as a separate layer** — I deliberately kept `distilled_memory` as a separate table that is *read* at session start and *written* by the offline consolidation job, never mutated inline during a turn. This matches the sleep-phase consolidation model: the base agent doesn't change, only the retrieval context does.

**Coordination approach** — I assigned each teammate a module boundary: Cherluk owned curriculum + wake-phase episodes, Olu owned tools + safety + memory protection, Taki owned RL policy + evaluation + memory testing. I owned the orchestrator, schema, and integration. No merge conflicts because no one touched another person's tables or files.

**What I learned:**

Building an AI system from scratch taught me that the architecture decisions made at the start (clean interfaces, typed return values, single-file DB, protocol-driven memory) determined how easy or hard integration was later. The 4-step memory protocol looks clean now — but it only looks clean because each step was designed with a clear input, output, and owner from day one.

---

## 🏁 Project Complete

**SHAPE Agent 16** is a fully functional interactive learning companion that:
- Teaches Python and AI concepts with an LLM-backed multi-agent system
- Adapts to individual learner levels using a contextual bandit RL policy
- Executes and debugs student code in a sandboxed environment
- Lets any team member publish curriculum without coding
- Wraps the learning experience in a full Python RPG battle game
- Maintains durable cross-session memory via episodic encoding and offline consolidation

Every component was built, verified, and documented by a named team member.

| Step | Owner | File |
|------|-------|------|
| Step 1 — Wake Phase | Cherluk | `shape_memory.py` |
| Step 2 — Consolidation | Michael | `shape_consolidate.py` |
| Step 3 — Recall Probes | Taki | `shape_recall_probe.py` |
| Step 4 — Stability Check | Taki + Michael | `shape_stability_check.py` |

---
*SHAPE Agent 16 — ITAI2376_14562 | **Michael Rivas** · Cherluk · Olu · Taki*